# Federated DTI prediction on Colab

Runs the centralized baseline, the local-only baseline or FedAvg (MPNN-CNN on DAVIS) on a Colab GPU.
Checkpoints and metrics are written to Google Drive after every epoch / round.

1. **Runtime → Change runtime type → GPU.**
2. Set the config cells below, then **Runtime → Run all**.
3. If Colab disconnects, reconnect and **Run all** again with the same `RUN_NAME`: training resumes from the last checkpoint on Drive.

To change hyperparameters, use a new `RUN_NAME` (the scripts refuse to resume a run with different settings). `EPOCHS` / `ROUNDS` may be increased to extend a finished run.

## 1. Config

In [ ]:
# ---- Repository ----
REPO_URL = "https://github.com/YOUR_USERNAME/federated-deeppurpose.git"  # <- set this
BRANCH = "master"

# ---- Output on Google Drive ----
DRIVE_DIR = "/content/drive/MyDrive/federated-deeppurpose"

# ---- Experiment ----
EXPERIMENT = "fedavg"            # "centralized" | "local" | "fedavg"
RUN_NAME = "fedavg_r30_e2_s42"   # one folder per run; reuse the name to resume
SEED = 42

In [ ]:
# ---- Hyperparameters ----
EPOCHS = 100        # centralized / local
ROUNDS = 30         # fedavg
LOCAL_EPOCHS = 2    # fedavg
BATCH_SIZE = 128
LR = 1e-3
NUM_WORKERS = 2
SUBSAMPLE = None    # e.g. 0.05 for a quick test run

## 2. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

For a private repository, use `https://<token>@github.com/...` as `REPO_URL` (a GitHub personal access token with read access).

In [ ]:
REPO_DIR = "/content/federated-deeppurpose"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

## 3. Install dependencies

Colab's own torch / numpy / pandas / scipy / scikit-learn / matplotlib / tensorboard are pinned with a constraints
file so pip cannot replace them. DeepPurpose is installed with `--no-deps` because its requirements pull in
`ax-platform` and `dgllife`, which MPNN-CNN does not use and which would reinstall torch.

In [ ]:
!pip list --format=freeze | grep -iE "^(torch|torchvision|torchaudio|numpy|pandas|scipy|scikit-learn|matplotlib|tensorboard)==" > /content/constraints.txt
!cat /content/constraints.txt
!pip install -q -r requirements_colab.txt -c /content/constraints.txt
!pip install -q --no-deps DeepPurpose==0.1.5

In [ ]:
import torch, numpy, pandas
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
from DeepPurpose import DTI  # fails here if a dependency is missing
if not torch.cuda.is_available():
    print("WARNING: running on CPU. Runtime -> Change runtime type -> GPU")

Optional: check that the cached protein encoding matches DeepPurpose exactly on this machine (about a minute).

In [ ]:
# !python tests/verify_protein_cache.py --n 1024

## 4. Train

In [ ]:
OUT = f"{DRIVE_DIR}/outputs/{RUN_NAME}"
args = (f'--output-dir "{OUT}" --seed {SEED} --batch-size {BATCH_SIZE} --lr {LR} '
        f'--num-workers {NUM_WORKERS} --resume')
if SUBSAMPLE:
    args += f" --subsample {SUBSAMPLE}"

script = {
    "centralized": f"experiments/run_centralized.py --epochs {EPOCHS}",
    "local": f"experiments/run_local.py --epochs {EPOCHS}",
    "fedavg": f"experiments/run_federated.py --rounds {ROUNDS} --local-epochs {LOCAL_EPOCHS}",
}[EXPERIMENT]

cmd = f"python {script} {args}"
print(cmd)
!{cmd}

## 5. Results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

if EXPERIMENT == "local":
    display(pd.read_csv(f"{OUT}/local_summary.csv").round(4))
else:
    m = pd.read_csv(f"{OUT}/metrics.csv")
    step = m.columns[0]
    display(m.tail())
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for i, metric in enumerate(["mse", "ci"]):
        ax[i].plot(m[step], m[f"val_{metric}"], label="val")
        ax[i].plot(m[step], m[f"test_{metric}"], label="test")
        ax[i].set(xlabel=step, ylabel=metric.upper())
        ax[i].legend()
    plt.tight_layout()
    plt.show()

    final = f"{OUT}/final_metrics.json"
    if os.path.exists(final):
        with open(final) as f:
            result = json.load(f)
        print("Best model, global test:", json.dumps(result["test"]["all"], indent=2))